# HarmonicNet (C4) Architecture
This notebook contains the implementation of the HarmonicNet baseline.
It is an equivariant U-Net but relies on a standard C4 discrete group (4 rotations) instead of SE(2) N=8.


In [ ]:
import torch
import torch.nn as nn
from escnn import gspaces
import escnn.nn as enn

class _EqConv(nn.Module):
    def __init__(self, in_t, out_t, mid_t=None):
        super().__init__()
        if mid_t is None: mid_t = out_t
        self.seq = enn.SequentialModule(
            enn.R2Conv(in_t, mid_t, 3, padding=1, bias=False),
            enn.InnerBatchNorm(mid_t), enn.ReLU(mid_t, inplace=True),
            enn.R2Conv(mid_t, out_t, 3, padding=1, bias=False),
            enn.InnerBatchNorm(out_type=out_t), enn.ReLU(out_t, inplace=True),
        )
    def forward(self, x): return self.seq(x)

class _EqDown(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.pool = enn.PointwiseMaxPool(a, 2)
        self.conv = _EqConv(a, b)
    def forward(self, x): return self.conv(self.pool(x))

class _EqUp(nn.Module):
    def __init__(self, a, b):
        super().__init__()
        self.up   = enn.R2Upsampling(a, scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = _EqConv(a + b, b)
    def forward(self, x1, x2):
        return self.conv(enn.tensor_directsum([x2, self.up(x1)]))

class HarmonicNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=2, N=4, base_channels=32):
        super().__init__()
        self.act = gspaces.rot2dOnR2(N=N)
        c = base_channels
        def ft(n): return enn.FieldType(self.act, n * [self.act.regular_repr])
        self.fin = enn.FieldType(self.act, n_channels * [self.act.trivial_repr])
        f1, f2, f3, f4, f5 = ft(c), ft(c*2), ft(c*4), ft(c*8), ft(c*16)
        self.inc   = _EqConv(self.fin, f1)
        self.d1, self.d2, self.d3, self.d4 = _EqDown(f1,f2), _EqDown(f2,f3), _EqDown(f3,f4), _EqDown(f4,f5)
        self.u1, self.u2, self.u3, self.u4 = _EqUp(f5,f4), _EqUp(f4,f3), _EqUp(f3,f2), _EqUp(f2,f1)
        out_t = enn.FieldType(self.act, n_classes * [self.act.trivial_repr])
        self.outc = enn.R2Conv(f1, out_t, 1)

    def forward(self, x):
        g = enn.GeometricTensor(x, self.fin)
        x1=self.inc(g); x2=self.d1(x1); x3=self.d2(x2); x4=self.d3(x3); x5=self.d4(x4)
        x=self.u1(x5,x4); x=self.u2(x,x3); x=self.u3(x,x2); x=self.u4(x,x1)
        return self.outc(x).tensor


### Model Initialization
Here is how to initialize HarmonicNet:


In [ ]:
model = HarmonicNet(n_channels=3, n_classes=2, N=4, base_channels=32)
print("HarmonicNet instantiated successfully!")
